# Introduction to PyTorch Notebook

> Hands-on Build It and Exercises.

## Build It

A 3-layer MLP trained on MNIST using only PyTorch primitives. No high-level wrappers. No `torchvision.datasets`. We download and parse the raw data ourselves.

### Step 1: Load MNIST From Raw Files

MNIST ships as 4 gzipped files: training images (60,000 x 28 x 28), training labels, test images (10,000 x 28 x 28), test labels. We download them and parse the binary format.

In [ ]:
```python

import torch

import torch.nn as nn

import struct

import gzip

import urllib.request

import os

def download_mnist(path="./mnist_data"):

    base_url = "https://storage.googleapis.com/cvdf-datasets/mnist/"

    files = [

        "train-images-idx3-ubyte.gz",

        "train-labels-idx1-ubyte.gz",

        "t10k-images-idx3-ubyte.gz",

        "t10k-labels-idx1-ubyte.gz",

    ]

    os.makedirs(path, exist_ok=True)

    for f in files:

        filepath = os.path.join(path, f)

        if not os.path.exists(filepath):

            urllib.request.urlretrieve(base_url + f, filepath)

def load_images(filepath):

    with gzip.open(filepath, "rb") as f:

        magic, num, rows, cols = struct.unpack(">IIII", f.read(16))

        data = f.read()

        images = torch.frombuffer(bytearray(data), dtype=torch.uint8)

        images = images.reshape(num, rows * cols).float() / 255.0

    return images

def load_labels(filepath):

    with gzip.open(filepath, "rb") as f:

        magic, num = struct.unpack(">II", f.read(8))

        data = f.read()

        labels = torch.frombuffer(bytearray(data), dtype=torch.uint8).long()

    return labels

In [ ]:
```

### Step 2: Define the Model

A 3-layer MLP: 784 -> 256 -> 128 -> 10. ReLU activations. Dropout for regularization. No batch norm to keep it simple.

In [ ]:
```python

class MNISTModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(784, 256),

            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(256, 128),

            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(128, 10),

        )

    def forward(self, x):

        return self.net(x)

In [ ]:
```

The output layer produces 10 raw logits (one per digit). No softmax -- `CrossEntropyLoss` handles that internally.

Parameter count: 784*256 + 256 + 256*128 + 128 + 128*10 + 10 = 235,146. Tiny by modern standards. GPT-2 small has 124M. This trains in seconds.

### Step 3: Training Loop

The canonical forward-loss-backward-step pattern.

In [ ]:
```python

def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    total_loss = 0

    correct = 0

    total = 0

    for images, labels in loader:

        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item() * images.size(0)

        _, predicted = outputs.max(1)

        correct += predicted.eq(labels).sum().item()

        total += labels.size(0)

    return total_loss / total, correct / total

def evaluate(model, loader, criterion, device):

    model.eval()

    total_loss = 0

    correct = 0

    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images, labels = images.to(device), labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)

            _, predicted = outputs.max(1)

            correct += predicted.eq(labels).sum().item()

            total += labels.size(0)

    return total_loss / total, correct / total

In [ ]:
```

Note `torch.no_grad()` during evaluation. This disables autograd, reducing memory usage and speeding up inference. Without it, PyTorch builds a computational graph you never use.

### Step 4: Wire Everything Together

In [ ]:
```python

def main():

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    download_mnist()

    train_images = load_images("./mnist_data/train-images-idx3-ubyte.gz")

    train_labels = load_labels("./mnist_data/train-labels-idx1-ubyte.gz")

    test_images = load_images("./mnist_data/t10k-images-idx3-ubyte.gz")

    test_labels = load_labels("./mnist_data/t10k-labels-idx1-ubyte.gz")

    train_dataset = torch.utils.data.TensorDataset(train_images, train_labels)

    test_dataset = torch.utils.data.TensorDataset(test_images, test_labels)

    train_loader = torch.utils.data.DataLoader(

        train_dataset, batch_size=64, shuffle=True

    )

    test_loader = torch.utils.data.DataLoader(

        test_dataset, batch_size=256, shuffle=False

    )

    model = MNISTModel().to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    num_params = sum(p.numel() for p in model.parameters())

    print(f"Device: {device}")

    print(f"Parameters: {num_params:,}")

    print(f"Train samples: {len(train_dataset):,}")

    print(f"Test samples: {len(test_dataset):,}")

    print()

    for epoch in range(10):

        train_loss, train_acc = train_one_epoch(

            model, train_loader, criterion, optimizer, device

        )

        test_loss, test_acc = evaluate(

            model, test_loader, criterion, device

        )

        print(

            f"Epoch {epoch+1:2d} | "

            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "

            f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}"

        )

    torch.save(model.state_dict(), "mnist_mlp.pt")

    print(f"\nModel saved to mnist_mlp.pt")

    print(f"Final test accuracy: {test_acc:.4f}")

In [ ]:
```

Expected output after 10 epochs: ~97.8% test accuracy. Training time on CPU: ~30 seconds. On GPU: ~5 seconds. On your mini framework with the same architecture: ~45 minutes.

## Exercises

In [ ]:
1. **Add batch normalization.** Insert `nn.BatchNorm1d` after each linear layer (before the activation). Compare test accuracy and training speed vs the dropout-only version. Batch norm should reach 98%+ in fewer epochs.

2. **Implement a learning rate finder.** Train for one epoch with exponentially increasing learning rate (from 1e-7 to 1.0). Plot loss vs LR. The optimal LR is just before the loss starts climbing. Use this to pick a better LR for the MNIST model.

3. **Port to GPU with mixed precision.** Add `torch.amp.autocast` and `GradScaler` to the training loop. Measure throughput (samples/second) with and without mixed precision on GPU. On an A100, expect ~2x speedup.

4. **Build a custom Dataset.** Download Fashion-MNIST (same format as MNIST but with clothing items). Implement a `FashionMNISTDataset(Dataset)` class with `__getitem__` and `__len__`. Train the same MLP and compare accuracy. Fashion-MNIST is harder -- expect ~88% vs ~98%.

5. **Replace Adam with SGD + momentum.** Train with `SGD(params, lr=0.01, momentum=0.9)`. Compare convergence curves. Then add a `CosineAnnealingLR` scheduler and see if SGD catches up to Adam by epoch 10.